# Casey coarse-class confidence sweep on CAVE K=10

Compare Mean, Pointwise-MLP, and GT with mixed-cell batches on held-out fold 0. Every run uses the CAVE K=10 windows with the same native cohort labels and fold assignments. Casey coarse-label confidence cutoffs are 0, 0.3, 0.5, 0.7, and 0.9. Neurons are matched to Casey by root ID or stable cell identity; ChC and L6b are excluded. The L2 targets are pyramidal, thalamocortical, BasketFam, MartFam, BipFam, and NglFam.

All 30 extended-axon thalamocortical cells use the source-manifest label because Casey provides no coarse confidence score for them. Each held-out fold contains six thalamocortical cells. Folds are balanced by Casey fine type within each confidence cohort; a cell keeps the same held-out fold across cutoffs. The test population changes with the confidence cutoff.


In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "results" / "presynaptic").is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from analysis.presynaptic.casey_confidence_cave.casey_confidence_comparison import (
    COHORT_ROOT, RESULT_ROOT, coarse_split_matrix, load_coarse_cohort_counts, load_summaries, load_training_curves,
    plot_confusions, plot_metric_comparison, plot_training_curves,
)
RESULT_ROOT

PosixPath('/orcd/home/002/jcbliao/rotation/segclr/gnn_classifier/results/presynaptic/casey_confidence_cave/k10')

## Cohort sizes by Casey coarse type

Train and test cell counts by Casey coarse type in the selected held-out fold after each confidence cutoff. Counts are planned from cell metadata until the CAVE cohorts are prepared, then read from the finalized fold manifests. Thalamocortical appears as a separate source-manifest type.


In [2]:
from analysis.presynaptic.casey_confidence_cave import casey_confidence_comparison as confidence_analysis
FOLD = globals().get("FOLD", 0)  # Held-out fold; change to 0–4.
cohorts = confidence_analysis.load_coarse_cohort_counts(fold=FOLD)
if cohorts.empty:
    print(f"No cohort files found under {confidence_analysis.COHORT_ROOT}")
else:
    if cohorts.attrs.get("planned"):
        print("Planned cell splits from Casey labels; CAVE cohorts are not finalized.")
    display(cohorts.sort_values(["confidence", "coarse_type"]).reset_index(drop=True))


,confidence,coarse_type,train_cells,test_cells
0,0.0,BasketFam,136,35
1,0.0,BipFam,91,22
2,0.0,L2,175,44
3,0.0,L3,198,49
4,0.0,L4,428,107
...,...,...,...,...
60,0.9,L6CT,142,35
61,0.9,L6IT,73,20
62,0.9,MartFam,80,20
63,0.9,NglFam,15,3


## Cell splits by Casey coarse type

Rows are Casey coarse types for the selected held-out fold. Each confidence level has **Train** and **Test** columns containing cell counts. Thalamocortical has no Casey coarse label, so its source-manifest type appears as a separate row. The five folds are balanced by Casey fine type within each confidence cohort.


In [3]:
from analysis.presynaptic.casey_confidence_cave import casey_confidence_comparison as confidence_analysis
FOLD = globals().get("FOLD", 0)  # Held-out fold; change to 0–4.
coarse_cohorts = confidence_analysis.load_coarse_cohort_counts(fold=FOLD)
if coarse_cohorts.empty:
    print(f"No cohort files found under {confidence_analysis.COHORT_ROOT}")
else:
    if coarse_cohorts.attrs.get("planned"):
        print("Planned cell splits from Casey labels; CAVE cohorts are not finalized.")
    matrix = confidence_analysis.coarse_split_matrix(coarse_cohorts)
    display(
        matrix.style
        .set_caption(f"Cell counts by Casey coarse type and minimum confidence — fold {FOLD}")
        .format("{:d}")
        .set_table_styles([
            {"selector": "th", "props": [("background-color", "#edf2f7"), ("text-align", "center")]},
            {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
        ])
    )


## Training progress

Curves appear as soon as a mixed-cell run writes its first epoch. Every retained window is used once per epoch.

In [4]:
curves = load_training_curves()
if curves.empty:
    print("No completed epochs yet.")
else:
    display(curves.groupby(["confidence", "model", "run"], observed=True).tail(1)
            .sort_values(["confidence", "model"]))
    for fig in plot_training_curves(curves):
        display(fig)
        plt.close(fig)

No completed epochs yet.


## Best-checkpoint test metrics

Each point uses the checkpoint selected by the highest **window macro F1**. The selected epoch is labeled on the first panel. Cell macro F1 is evaluated at that same checkpoint, so it need not equal the maximum cell macro F1 visible in the training curves. The default experiment uses fold 0. If additional folds are explicitly run, error bars show their standard deviation.


In [5]:
summary, payloads = load_summaries()
if summary.empty:
    print("Final summaries appear after training finishes.")
else:
    display(summary)
    fig = plot_metric_comparison(summary)
    display(fig)
    plt.close(fig)

Final summaries appear after training finishes.


## Confusion matrices

Rows are true classes, columns are predictions. Each row is normalized to sum to one.

In [6]:
for scope in ("window", "cell"):
    for fig in plot_confusions(payloads, scope=scope, normalize=True):
        display(fig)
        plt.close(fig)